# 🏥 TRICARE RAG Q&A 통합 파이프라인

## 개요
- `tricare_rag_pipeline.ipynb`와 `tricare_exclusion.ipynb`를 하나로 통합한 노트북임
- 단계별로 순서대로 실행하면 전체 RAG Q&A 시스템이 구동됨
- 팀원 벡터 DB와 나중에 병합할 수 있도록 경로·컬렉션명을 `tricare_` 프리픽스로 분리 관리함

## 사용 데이터
| 파일 | 형식 | 저장소 | 설명 |
|---|---|---|---|
| `Costs_Fees.pdf` | PDF | `chroma_db` (텍스트) + `chroma_db2` (표) | TRICARE 2026 비용·수수료 전체 표 |
| `TOP_Handbook_AUG_2023_FINAL.pdf` | PDF | `chroma_db` (텍스트) | TRICARE Overseas Program 핸드북 |
| `mental_health_services.csv` | CSV | `chroma_db` (텍스트) | 정신건강 서비스별 보장 내용 |
| `Health_Plan_Costs.csv` | CSV | `chroma_db2` (표) | 플랜별 비용 수치 표 |
| `TricarePlans.csv` | CSV | `chroma_db` (텍스트) | 플랜별 특징·비교 정보 |
| `tricare_exclusions.csv` | CSV | `chroma_db` (텍스트) | 보장 제외 항목 (크롤러 대체) |

## 사용 모델 / 도구
| 구성 요소 | 선택 |
|---|---|
| LLM | `gpt-4o-mini` |
| 임베딩 | `BAAI/bge-m3` (다국어 100개+) |
| 벡터 DB | Chroma (로컬) |
| PDF 텍스트 | PyMuPDFLoader |
| PDF 표 추출 | pdfplumber |

## 전체 실행 순서
1. 패키지 설치 → 환경 설정 → 파일 경로 설정
2. 임베딩 모델 초기화
3. PDF 로드 → 청킹
4. CSV 로드 → Document 변환 → 청킹 (mental_health / TricarePlans / tricare_exclusions)
5. 텍스트+CSV 통합 벡터 저장소 생성 (`chroma_db`)
6. PDF 표 추출 + Health_Plan_Costs.csv → 표 전용 벡터 저장소 생성 (`chroma_db2`)
7. LLM + RAG 체인 구성 (다국어 답변 지원)
8. Demo 실행 (일반 LLM 비교 / 텍스트 RAG / 표 RAG)

## 벡터 저장소 구조
```
chroma_db  (tricare_rag)
  ├── Costs_Fees.pdf 텍스트 청크
  ├── TOP_Handbook.pdf 텍스트 청크
  ├── mental_health_services.csv
  ├── TricarePlans.csv          ← 신규 추가
  └── tricare_exclusions.csv   ← 크롤러 대체

chroma_db2 (tricare_cost_tables)
  ├── Costs_Fees.pdf 표 (pdfplumber)
  └── Health_Plan_Costs.csv    ← 신규 추가
```


---
## Section 0 · 패키지 설치

### 개요
- 처음 실행 시에만 필요함
- 주석 해제 후 실행하면 됨


In [ ]:
# !pip install langchain langchain-community langchain-chroma langchain-openai \
# pymupdf pdfplumber sentence-transformers chromadb python-dotenv -q

---
## Section 1 · 환경 설정

### 개요
- `.env` 파일에서 `OPENAI_API_KEY`를 읽어옴
- 벡터 DB 경로와 컬렉션명을 상수로 분리해 관리함
- 나중에 팀 통합 벡터 DB를 만들 때 경로만 참고하면 됨

### 팀별 벡터 DB 경로 (통합 시 참고)
| 팀원 | persist_directory | collection_name |
|------|------------------|-----------------|
| TRICARE (본인) | `./chroma_db`, `./chroma_db2` | `tricare_rag`, `tricare_cost_tables` |
| Cigna (은우님) | `./chroma_cigna_latest`, `./chroma_cigna_all` | `cigna_latest`, `cigna_all` |
| Allianz (수진님) | `./vectordb` | `allianz_care` |


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('OPENAI_API_KEY')
print(f"✅ API Key 로드: {'성공' if api_key else '실패 → .env 파일 확인 필요'}")

# 벡터 DB 경로·컬렉션명 상수
# 팀 통합 벡터 DB 병합 시 이 상수들을 기준으로 경로를 맞춤
PERSIST_TEXT  = './chroma_db'
PERSIST_TABLE = './chroma_db2'
COLLECTION_TEXT  = 'tricare_rag'
COLLECTION_TABLE = 'tricare_cost_tables'


---
## Section 2 · 파일 경로 설정

### 개요
- PDF 2개 + CSV 4개 총 6개 데이터 소스 경로를 설정함
- 파일 존재 여부를 ✅/❌로 한 번에 확인함


In [ ]:
# data 폴더에 아래 데이터 다 있음
# 실행 시 본인 로컬 위치에 맞게 절대 경로 설정할 것을 추천함
PDF_COSTS       = 'C:\SKN-_24\제 3차 프로젝트\data\Costs_Fees.pdf'
PDF_HANDBOOK    = 'C:\SKN-_24\제 3차 프로젝트\data\TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf'
CSV_MENTAL      = 'C:\SKN-_24\제 3차 프로젝트\data\mental_health_services.csv'
CSV_COSTS       = 'C:\SKN-_24\제 3차 프로젝트\data\Health_Plan_Costs.csv'
CSV_PLANS       = 'C:\SKN-_24\제 3차 프로젝트\data\TricarePlans.csv'
CSV_EXCLUSIONS  = 'C:\SKN-_24\제 3차 프로젝트\data\tricare_exclusions.csv'

for path in [PDF_COSTS, PDF_HANDBOOK, CSV_MENTAL, CSV_COSTS, CSV_PLANS, CSV_EXCLUSIONS]:
    status = '✅' if os.path.exists(path) else '❌'
    print(f'{status} {path}')


---
## Section 3 · 임베딩 모델 초기화 — `BAAI/bge-m3`

### 개요
- 100개 이상의 언어를 동일한 벡터 공간에 매핑하는 다국어 임베딩 모델임
- 한국어 질문과 영어 문서 간 유사도 매칭이 가능함
- 팀원 Cigna 코드와 동일한 모델을 사용해 임베딩 방식을 통일함


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cpu'}  # GPU 있으면 'cuda'로 변경
)

print('✅ BAAI/bge-m3 임베딩 모델 로드 완료')


---
## Section 4 · PDF 로드 — PyMuPDFLoader

### 개요
- `PyPDFLoader` 대신 `PyMuPDFLoader`를 사용함
  - 압축된 최신 PDF에서 파싱 에러가 발생하는 경우가 있음
  - `PyMuPDFLoader`는 동일한 페이지 단위 추출 방식이지만 안정성이 더 높음
- 두 PDF를 로드한 뒤 하나의 리스트로 합쳐서 청킹 진행
- `metadata.source` 필드로 청크 출처 추적 가능함


In [ ]:
# !pip install pymupdf

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

loader_costs    = PyMuPDFLoader(PDF_COSTS)
loader_handbook = PyMuPDFLoader(PDF_HANDBOOK)

docs_costs    = loader_costs.load()
docs_handbook = loader_handbook.load()

all_pdf_docs = docs_costs + docs_handbook

print(f'1. Costs_Fees.pdf: {len(docs_costs)}페이지')
print(f'2. TOP_Handbook.pdf: {len(docs_handbook)}페이지')
print(f'=> 합계: {len(all_pdf_docs)}페이지')
print(f'\n--- 샘플 메타데이터 ---')
print(all_pdf_docs[0].metadata)


---
## Section 5 · PDF 청킹 — RecursiveCharacterTextSplitter

### 개요
- `chunk_size=500`: 보험 약관은 문장이 길고 맥락이 이어지는 경우가 많아 기본값(200)보다 크게 설정함
- `chunk_overlap=50`: 청크 경계에서 문맥이 끊기는 것을 방지함
- 이 `text_splitter`는 이후 모든 CSV 청킹에도 재사용됨


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ', '']
)

pdf_chunks = text_splitter.split_documents(all_pdf_docs)

print(f'분할 결과: {len(all_pdf_docs)}페이지 → {len(pdf_chunks)}개 청크')
print(f'\n--- 청크 샘플 ---')
print(pdf_chunks[0].page_content[:300])
print(f'\nMetadata: {pdf_chunks[0].metadata}')


---
## Section 6 · CSV 로드 — 텍스트 벡터 저장소용 3종

### 개요
- 텍스트 벡터 저장소(`chroma_db`)에 들어갈 CSV 3종을 Document로 변환함
- `CSVLoader` 대신 직접 변환 방식을 사용함
  - 청킹 후에도 서비스명·항목명이 각 청크에 남아 출처 추적이 가능함

### 3종 파일별 변환 방식
| 파일 | page_content 구성 | metadata 키 |
|---|---|---|
| `mental_health_services.csv` | `서비스: {서비스}\n내용: {내용}` | `source`, `서비스명`, `type` |
| `TricarePlans.csv` | `항목: {Plans}\n플랜명: {플랜명}\n내용: {내용}` | `source`, `plan_name`, `type` |
| `tricare_exclusions.csv` | `제외항목: {name}\n내용: {content}` | `source`, `exclusion_name`, `type`, `url` |


In [ ]:
import pandas as pd
from langchain_core.documents import Document

# 1. mental_health_services.csv
# 서비스명을 page_content에 직접 포함시켜 청킹 후에도 서비스명이 남도록 함
df_mental = pd.read_csv(CSV_MENTAL, encoding='utf-8-sig')
mental_docs = []
for _, row in df_mental.iterrows():
    mental_docs.append(Document(
        page_content=f"서비스: {row['서비스']}\n내용: {row['내용']}",
        metadata={
            'source': 'mental_health_services.csv',
            '서비스명': row['서비스'],
            'type': 'mental_health'
        }
    ))

mental_chunks = text_splitter.split_documents(mental_docs)
print(f'📋 mental_health_services: {len(df_mental)}행 → {len(mental_chunks)}청크')


In [ ]:
# 2. TricarePlans.csv — 신규 추가
# 컬럼 구조: Plans(행 항목) + 각 플랜명(열)
# 플랜별로 항목-내용 쌍을 Document로 변환해 플랜 비교 질문에 검색되도록 함
df_plans = pd.read_csv(CSV_PLANS, encoding='utf-8-sig')
plan_cols = [c for c in df_plans.columns if c != 'Plans']
plans_docs = []
for _, row in df_plans.iterrows():
    topic = row['Plans']
    for plan_name in plan_cols:
        content = str(row[plan_name]).strip()
        if not content or content == 'nan':
            continue
        plans_docs.append(Document(
            page_content=f"항목: {topic}\n플랜명: {plan_name}\n내용: {content}",
            metadata={
                'source': 'TricarePlans.csv',
                'plan_name': plan_name,
                'type': 'plan_info'
            }
        ))

plans_chunks = text_splitter.split_documents(plans_docs)
print(f'📋 TricarePlans: {len(df_plans)}행 × {len(plan_cols)}플랜 → {len(plans_chunks)}청크')


In [ ]:
# 3. tricare_exclusions.csv — 크롤러(tricare_exclusions_crawler) 대체
# 기존에는 크롤러를 실행해 실시간으로 가져왔으나 CSV로 대체함
# content 컬럼에 보장 제외 이유와 상세 설명이 포함되어 있음
# url은 metadata에만 저장해 출처 추적에 활용함
df_excl = pd.read_csv(CSV_EXCLUSIONS, encoding='utf-8-sig')
exclusion_docs = []
for _, row in df_excl.iterrows():
    exclusion_docs.append(Document(
        page_content=f"제외항목: {row['name']}\n내용: {row['content']}",
        metadata={
            'source': 'tricare_exclusions.csv',
            'exclusion_name': row['name'],
            'type': 'exclusion',
            'url': row['url']
        }
    ))

exclusion_chunks = text_splitter.split_documents(exclusion_docs)
print(f'📋 tricare_exclusions: {len(df_excl)}행 → {len(exclusion_chunks)}청크')


---
## Section 7 · 텍스트+CSV 통합 벡터 저장소 생성 (`chroma_db`)

### 개요
- PDF 텍스트 + CSV 3종(정신건강 / 플랜 정보 / 제외 항목)을 하나의 저장소에 통합함
- Retriever가 소스 구분 없이 가장 관련 있는 청크를 한 번에 검색함
- `metadata.source`와 `metadata.type`으로 청크 출처는 항상 추적 가능함

### 통합 청크 구성
| 소스 | 청크 변수 |
|---|---|
| PDF 텍스트 2종 | `pdf_chunks` |
| mental_health_services.csv | `mental_chunks` |
| TricarePlans.csv | `plans_chunks` |
| tricare_exclusions.csv | `exclusion_chunks` |


In [ ]:
from langchain_chroma import Chroma

# PDF + CSV 3종 모두 합산
all_text_chunks = pdf_chunks + mental_chunks + plans_chunks + exclusion_chunks

print(f'📦 청크 합산')
print(f'   PDF 텍스트:             {len(pdf_chunks)}개')
print(f'   mental_health_services: {len(mental_chunks)}개')
print(f'   TricarePlans:           {len(plans_chunks)}개')
print(f'   tricare_exclusions:     {len(exclusion_chunks)}개')
print(f'   합계:                   {len(all_text_chunks)}개')
print('⏳ 벡터 저장소 생성 중...')

# PERSIST_TEXT / COLLECTION_TEXT는 Section 1에서 정의한 상수
vector_store = Chroma.from_documents(
    documents=all_text_chunks,
    embedding=embedding_model,
    collection_name=COLLECTION_TEXT,
    persist_directory=PERSIST_TEXT
)

print(f'✅ 텍스트 벡터 저장소 생성 완료: {vector_store._collection.count()}개 벡터')


---
## Section 8 · 표 데이터 수집 — pdfplumber + `Health_Plan_Costs.csv`

### 개요
- 비용 수치 전용 저장소(`chroma_db2`)에 들어갈 데이터를 두 가지 방식으로 수집함
- **pdfplumber**: `Costs_Fees.pdf`에서 표를 추출함. 행/열 구조를 `|` 구분자로 보존함
- **`Health_Plan_Costs.csv`**: 플랜별 비용 수치를 직접 로드함 (신규 추가)
  - `탭` 컬럼: 플랜명·그룹 정보 (예: `TRICARE Prime & TRICARE Select Group A`)
  - `비용 유형` 컬럼: 비용 항목 (예: `Deductible`, `Enrollment fee`)
  - `플랜 1`, `플랜 2` 컬럼: 각 플랜의 수치
  - 두 소스를 함께 넣으면 같은 수치를 서로 다른 형태로 검색할 수 있어 정확도가 올라감


In [ ]:
import pdfplumber

def extract_tables_from_pdf(pdf_path):
    """
    pdfplumber로 PDF 전체에서 표를 추출함.
    각 셀은 ' | '로 구분되어 LLM이 행/열 관계를 파악할 수 있음.
    """
    table_docs = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            for table_idx, table in enumerate(tables):
                if not table or len(table) < 2:
                    continue
                lines = []
                for row in table:
                    cleaned = [
                        str(cell).strip().replace('\n', ' ') if cell else ''
                        for cell in row
                    ]
                    lines.append(' | '.join(cleaned))
                table_docs.append(Document(
                    page_content=f'[TRICARE 비용표 | p.{page_num + 1}]\n' + '\n'.join(lines),
                    metadata={
                        'source': pdf_path,
                        'page': page_num + 1,
                        'type': 'cost_table',
                        'table_idx': table_idx + 1
                    }
                ))
    return table_docs

table_docs_pdf = extract_tables_from_pdf(PDF_COSTS)
print(f'📊 PDF 표 추출: {len(table_docs_pdf)}개')
print('\n--- PDF 표 샘플 ---')
print(table_docs_pdf[0].page_content[:300])


In [ ]:
# Health_Plan_Costs.csv — 신규 추가
# 탭(플랜·그룹 정보) + 비용 유형 + 플랜 1/2 수치를 하나의 page_content로 구성
# pdfplumber 표와 동일한 저장소에 넣어 비용 질문의 검색 커버리지를 높임
df_costs = pd.read_csv(CSV_COSTS, encoding='utf-8-sig')
cost_docs = []
for _, row in df_costs.iterrows():
    tab       = str(row['탭']).strip()
    cost_type = str(row['비용 유형']).strip()
    plan1     = str(row['플랜 1']).strip()
    plan2     = str(row['플랜 2']).strip() if str(row['플랜 2']) != 'nan' else ''

    content = f"플랜/그룹: {tab}\n비용 유형: {cost_type}\n플랜 1: {plan1}"
    if plan2:
        content += f"\n플랜 2: {plan2}"

    cost_docs.append(Document(
        page_content=content,
        metadata={
            'source': 'Health_Plan_Costs.csv',
            'cost_type': cost_type,
            'type': 'cost_table'
        }
    ))

# Health_Plan_Costs는 행 자체가 이미 비용 단위로 구성되어 있어 추가 청킹 불필요
print(f'📊 Health_Plan_Costs: {len(df_costs)}행 → {len(cost_docs)}개 Document')
print('\n--- CSV 비용 샘플 ---')
print(cost_docs[0].page_content)


---
## Section 9 · 표 전용 벡터 저장소 생성 (`chroma_db2`)

### 개요
- PDF 표 + `Health_Plan_Costs.csv`를 하나의 표 전용 저장소에 통합함
- `k=3`: 표 데이터는 정보 밀도가 높아 3개면 충분함
- 비용 질문 시 텍스트 청크가 섞이지 않아 수치 답변 정확도가 높아짐

### 재실행 시 벡터 저장소 불러오는 방법
```python
vector_store = Chroma(
    collection_name='tricare_rag',
    embedding_function=embedding_model,
    persist_directory='./chroma_db'
)
table_vector_store = Chroma(
    collection_name='tricare_cost_tables',
    embedding_function=embedding_model,
    persist_directory='./chroma_db2'
)
```


In [ ]:
# PDF 표 + CSV 비용 Document 합산
all_table_docs = table_docs_pdf + cost_docs

print(f'📦 표 데이터 합산')
print(f'PDF 표 (pdfplumber): {len(table_docs_pdf)}개')
print(f'Health_Plan_Costs.csv: {len(cost_docs)}개')
print(f'합계: {len(all_table_docs)}개')
print('⏳ 표 전용 벡터 저장소 생성 중...')

# PERSIST_TABLE / COLLECTION_TABLE은 Section 1에서 정의한 상수
table_vector_store = Chroma.from_documents(
    documents=all_table_docs,
    embedding=embedding_model,
    collection_name=COLLECTION_TABLE,
    persist_directory=PERSIST_TABLE
)

print(f'✅ 표 전용 벡터 저장소 생성 완료: {table_vector_store._collection.count()}개 벡터')


---
## Section 10 · Retriever 설정

### 개요
- 텍스트+CSV 통합용 `retriever`와 표 전용 `table_retriever`를 각각 설정함
- `k=6`: Group A/B, TRICARE Prime/Select 등 같은 주제가 여러 청크에 분산되어 있어
  k=4면 한쪽 그룹 정보만 검색될 수 있음 → k=6으로 넉넉하게 설정함
- `k=3`: 표 데이터는 밀도가 높아 3개면 충분함


In [ ]:
retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 6}
)
table_retriever = table_vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

# 동작 확인
test_result = retriever.invoke('TRICARE Prime 본인부담금')
print(f'🔍 텍스트 retriever 테스트: {len(test_result)}개 검색됨')
for i, d in enumerate(test_result):
    print(f'  [{i+1}] {d.metadata.get("source","?")} | type={d.metadata.get("type","?")} : {d.page_content[:60]}...')


---
## Section 11 · LLM + RAG 체인 구성

### 개요
- `gpt-4o-mini`, `temperature=0.1`: 보험 정보 오답은 실제 피해로 이어질 수 있어
  낮은 temperature로 문서 기반 답변을 유도함
- `detect_language()`: 유니코드 범위로 한/중/일을 판별하고 나머지는 영어로 처리함
  - 팀원 A(rag_utils.py)와 동일한 방식으로 통일함
- `make_rag_chain()`: 질문마다 언어를 감지해 프롬프트에 `answer_language`를 f-string으로 주입함
  - `retriever`를 인자로 받아 텍스트용 / 표 전용 체인 모두 이 함수 하나로 생성 가능함
- `format_docs()`: 출처 정보(`[source, p.페이지]`)를 청크 앞에 붙여 반환함


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)

# 언어 감지 함수
# 유니코드 범위로 한/중/일을 판별하고 나머지는 영어로 처리
# 팀원 A(rag_utils.py)와 동일한 방식으로 다국어 답변 형식을 통일
def detect_language(text: str) -> str:
    if any('\u4e00' <= c <= '\u9fff' for c in text):
        return 'zh'
    if any('\u3040' <= c <= '\u30ff' for c in text):
        return 'ja'
    if any('\uac00' <= c <= '\ud7a3' for c in text):
        return 'ko'
    return 'en'

# 언어 코드 → 프롬프트용 언어명 매핑
# 'other'는 LLM이 질문 언어를 그대로 따라가도록 지시
LANGUAGE_NAME_MAP = {
    'ko': 'Korean',
    'en': 'English',
    'zh': 'Chinese',
    'ja': 'Japanese',
    'es': 'Spanish',
    'fr': 'French',
    'de': 'German',
    'other': 'the same language as the user\'s question',
}

# format_docs: 출처 정보를 청크 앞에 붙여 반환
# C팀원 RAG_TEMPLATE의 출처 인용 형식([source, p.페이지])과 통일
def format_docs(docs):
    result = []
    for doc in docs:
        source = doc.metadata.get('source', 'unknown')
        page   = doc.metadata.get('page', '')
        label  = f'[{source}{f", p.{page+1}" if page else ""}]'
        result.append(f'{label}{doc.page_content}')
    return '\n\n'.join(result)

# make_rag_chain: 질문마다 언어를 감지해 프롬프트에 answer_language를 f-string으로 주입
# retriever를 인자로 받아 텍스트용(retriever)·표 전용(table_retriever) 모두 사용 가능
# PromptTemplate 변수로 언어를 넘기면 체인 구조가 복잡해지므로 f-string 방식 사용
def make_rag_chain(question: str, retriever, model):
    language_code   = detect_language(question)
    answer_language = LANGUAGE_NAME_MAP.get(language_code, 'English')

    PROMPT_TEMPLATE = (
        'You are a TRICARE health benefits specialist.\n'
        'Answer questions based ONLY on the provided TRICARE documents below.\n'
        'If the answer is not in the documents, '
        'say "해당 내용은 제공된 문서에서 확인되지 않습니다." in the user\'s language.\n'
        'Always mention relevant conditions such as Group A/B, plan type, or beneficiary status.\n\n'
        f'IMPORTANT LANGUAGE RULE:\n'
        f'- You must answer in {answer_language}.\n'
        f'- The answer language must match the user\'s question language.\n'
        f'- Do not switch languages unless necessary for official document names or insurance terms.\n\n'
        '[참고 문서]\n{context}\n\n'
        '[질문]\n{question}\n\n'
        '[답변]\n'
    )

    rag_prompt = PromptTemplate.from_template(PROMPT_TEMPLATE)

    return (
        {'question': RunnablePassthrough(), 'context': retriever | format_docs}
        | rag_prompt
        | model
        | StrOutputParser()
    )

print('✅ LLM + RAG 체인 구성 완료')
print('   호출 방법: make_rag_chain(question, retriever, model).invoke(question)')


---
## Section 12 · Demo 1 — 일반 LLM vs RAG 비교

### 개요
- 같은 질문을 일반 LLM과 RAG에 각각 던져 답변 품질을 비교함
- 일반 LLM: 학습 데이터 기반 → 세션 종류별 시간 제한 등 세부 조건 누락 가능
- RAG: `mental_health_services.csv`에서 정확한 내용을 검색 후 답변
- 출력 형식: `QUESTION → ANSWER → RETRIEVED DOCS` 순서로 통일함


In [ ]:
from langchain_core.messages import HumanMessage

question = (
    'TRICARE Prime에서 정신건강 상담(Psychotherapy)은 어떻게 보장되나요? '
    '세션 종류별(개인/가족/그룹) 시간 제한과 주의사항도 알려주세요.'
)

print('=' * 100)
print('QUESTION:', question)

print('\n❓ [Demo 1 - 일반 LLM 답변] — 문서 없이 직접 응답')
print('-' * 65)
plain_response = model.invoke([HumanMessage(content=question)])
print(plain_response.content)
print('\n⚠️  주의: 위 답변은 학습 데이터 기반이며 실제 문서와 다를 수 있습니다.')


In [ ]:
# RAG 체인 생성 (질문 언어 감지 → 답변 언어 자동 설정)
rag_chain = make_rag_chain(question, retriever, model)

retrieved = retriever.invoke(question)
print('🔎 검색된 청크:')
for i, chunk in enumerate(retrieved):
    src = chunk.metadata.get('source', '')
    svc = chunk.metadata.get('서비스명', '')
    print(f'  [{i+1}] {src} {f"({svc})" if svc else ""}: {chunk.page_content[:80]}...')

print()
print('=== ANSWER ===')
rag_response = rag_chain.invoke(question)
print(rag_response)

print('\n=== RETRIEVED DOCS ===')
for doc in retrieved:
    print(doc.metadata)


---
## Section 13 · Demo 2 — 텍스트+CSV 통합 RAG

### 개요
- `retriever`(PDF + 정신건강 + 플랜 정보 + 제외 항목 통합)를 사용함
- `question` 변수만 바꿔서 다양한 질문을 테스트할 수 있음
- 질문 언어를 자동 감지해 같은 언어로 답변함

### 사용 방법
- 질문 바꾸고 싶을 때 → `question` 변수만 수정
- Demo 2만 쓰고 싶을 때 → Section 11까지 실행 후 이 셀만 실행
- Demo 2, 3 비교하고 싶을 때 → 두 셀 순서대로 실행


In [ ]:
question = '보험 가입 후 취소하고 싶을 때 전액 환불이 가능한 청약 철회 기간(Free Look Period)은 며칠인가요?'
# question = 'I plan on still working when I turn 65. How does this affect my TRICARE?'
# question = 'What is the TRICARE Prime Demo?'
# question = 'TRICARE Prime에서 침술(Acupuncture)은 보장이 되나요?'  # 제외 항목 테스트
# question = 'TRICARE Prime과 TRICARE Select의 차이는 무엇인가요?'     # 플랜 비교 테스트

print('=' * 100)
print('QUESTION:', question)

retrieved = retriever.invoke(question)
print('🔎 검색된 청크:')
for i, chunk in enumerate(retrieved):
    src  = chunk.metadata.get('source', '')
    typ  = chunk.metadata.get('type', '')
    print(f'  [{i+1}] {src} ({typ}): {chunk.page_content[:80]}...')

rag_chain = make_rag_chain(question, retriever, model)
answer = rag_chain.invoke(question)

print('=== ANSWER ===')
print(answer)

print('\n=== RETRIEVED DOCS ===')
for doc in retrieved:
    print(doc.metadata)


---
## Section 14 · Demo 3 — 표 전용 비용 Q&A

### 개요
- `table_retriever`(PDF 표 + Health_Plan_Costs.csv 통합)를 사용함
- 텍스트 청크가 섞이지 않아 `$26`, `$45` 같은 수치 답변이 더 정확함
- 복수 질문을 리스트로 넣어 한 번에 실행 가능함


In [ ]:
cost_questions = [
    'TRICARE Prime에서 Group A 재향군인 가족의 응급실 방문 비용은 얼마인가요?',
    'TRICARE Select Group B 재향군인의 네트워크 내 전문의 외래 진료비는 얼마인가요?',
    'TRICARE Reserve Select의 월 보험료(Member only)는 얼마인가요?'
]

for q in cost_questions:
    print('=' * 100)
    print('QUESTION:', q)

    # 질문마다 언어 감지 후 체인 생성
    table_rag_chain = make_rag_chain(q, table_retriever, model)
    answer = table_rag_chain.invoke(q)

    print('=== ANSWER ===')
    print(answer)

    print('\n=== RETRIEVED DOCS ===')
    for doc in table_retriever.invoke(q):
        print(doc.metadata)


---
## Section 15 · 정리 및 다음 단계

### 데이터 처리 전략 요약
| 데이터 | 처리 방식 | 저장소 |
|---|---|---|
| PDF 텍스트 | PyMuPDFLoader → 청킹 | `chroma_db` |
| PDF 표 | pdfplumber → Document | `chroma_db2` |
| mental_health_services.csv | 직접 row 변환 → 청킹 | `chroma_db` |
| TricarePlans.csv | 항목×플랜 쌍으로 변환 → 청킹 | `chroma_db` |
| tricare_exclusions.csv | 직접 row 변환 → 청킹 (크롤러 대체) | `chroma_db` |
| Health_Plan_Costs.csv | 직접 row 변환 (청킹 불필요) | `chroma_db2` |

### Demo 결과 요약
| | 일반 LLM | RAG 텍스트 | RAG 표 |
|---|---|---|---|
| 데이터 출처 | 학습 데이터 | PDF + 4종 CSV | PDF 표 + 비용 CSV |
| 정확도 | 낮음 | 높음 | 매우 높음 (수치) |
| 다국어 답변 | 가능 | ✅ 언어 감지 + 프롬프트 주입 | ✅ 언어 감지 + 프롬프트 주입 |

### 향후 개선 방향
1. **메타데이터 필터링** — `type`, `plan_name` 필터를 Retriever에 추가
2. **앙상블 Retriever** — BM25(키워드) + bge-m3(의미) 혼합 검색
3. **평가 파이프라인** — RAGAS 라이브러리로 답변 품질 정량 측정
4. **통합 벡터 DB 병합** — 팀원 각자 벡터 DB를 `add_documents()`로 통합 컬렉션에 병합
